# SpaceX Launch Records Dashboard — Completed Lab

Completed solution for the IBM Plotly Dash lab. This notebook implements the launch-site dropdown, success pie chart, payload range slider, scatter plot callback, and data-driven analysis. Keep `spacex_launch_dash.csv` in the same directory or allow the first cell to download it.

## Setup and load the dataset

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
from IPython.display import display

DATA_FILE = Path('spacex_launch_dash.csv')
DATA_URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv'
if not DATA_FILE.exists():
    pd.read_csv(DATA_URL).to_csv(DATA_FILE, index=False)
spacex_df = pd.read_csv(DATA_FILE)
required = ['Launch Site', 'class', 'Payload Mass (kg)', 'Booster Version', 'Booster Version Category']
missing = [c for c in required if c not in spacex_df.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')
spacex_df['class'] = pd.to_numeric(spacex_df['class'], errors='coerce')
spacex_df['Payload Mass (kg)'] = pd.to_numeric(spacex_df['Payload Mass (kg)'], errors='coerce')
spacex_df = spacex_df.dropna(subset=required[:3]).copy()
spacex_df['class'] = spacex_df['class'].astype(int)
display(spacex_df.head())

## Task 1 — Launch-site dropdown options

The dropdown defaults to `ALL`, contains every site, and is searchable in the Dash app.

In [ ]:
site_names = sorted(spacex_df['Launch Site'].dropna().unique())
dropdown_options = [{'label': 'All Sites', 'value': 'ALL'}] + [
    {'label': site, 'value': site} for site in site_names
]
dropdown_options

## Task 2 — Success pie-chart callback logic

The callback filters by the selected site. It visualizes class 1 and class 0 counts separately.

In [ ]:
def make_success_pie(data, selected_site='ALL'):
    filtered = data if selected_site == 'ALL' else data[data['Launch Site'] == selected_site]
    outcome = filtered['class'].map({0: 'Unsuccessful', 1: 'Successful'})
    counts = outcome.value_counts().reindex(['Successful', 'Unsuccessful'], fill_value=0)
    counts = counts.rename_axis('Launch Outcome').reset_index(name='Number of Launches')
    title = 'Launch Outcomes for All Sites' if selected_site == 'ALL' else f'Launch Outcomes for {selected_site}'
    fig = px.pie(counts, values='Number of Launches', names='Launch Outcome', title=title,
                 color='Launch Outcome', color_discrete_map={'Successful':'#2ca02c','Unsuccessful':'#d62728'})
    fig.update_traces(textposition='inside', textinfo='label+value+percent')
    return fig

display(make_success_pie(spacex_df, 'ALL'))

## Task 3 — Payload range slider

The slider minimum is 0 kg and the step is 1,000 kg. The upper limit is set to 10,000 kg or the rounded-up dataset maximum, whichever is larger.

In [ ]:
min_payload = max(0, int(spacex_df['Payload Mass (kg)'].min() // 1000 * 1000))
max_payload = max(10000, int(((spacex_df['Payload Mass (kg)'].max() + 999) // 1000) * 1000))
slider_config = {'min': 0, 'max': max_payload, 'step': 1000,
                 'value': [min_payload, max_payload],
                 'marks': {v: f'{v:,}' for v in range(0, max_payload + 1, 2000)}}
slider_config

## Task 4 — Payload-versus-outcome scatter callback logic

The callback filters both by launch site and payload range and colours points by booster-version category.

In [ ]:
def make_payload_scatter(data, selected_site='ALL', payload_range=(0, 10000)):
    low, high = payload_range
    filtered = data[data['Payload Mass (kg)'].between(low, high)].copy()
    if selected_site != 'ALL':
        filtered = filtered[filtered['Launch Site'] == selected_site]
    site_title = 'All Launch Sites' if selected_site == 'ALL' else selected_site
    fig = px.scatter(filtered, x='Payload Mass (kg)', y='class',
                     color='Booster Version Category',
                     hover_data=['Launch Site', 'Booster Version'],
                     title=f'Payload Mass vs. Launch Outcome — {site_title}',
                     labels={'class':'Launch Outcome (0 = Failure, 1 = Success)'})
    fig.update_yaxes(tickmode='array', tickvals=[0,1], ticktext=['Unsuccessful (0)','Successful (1)'])
    return fig

display(make_payload_scatter(spacex_df, 'ALL', (0, max_payload)))

## Run the interactive Dash application

The full app is supplied in `spacex_dash_app.py` alongside this notebook. Install dependencies with `pip install -r requirements.txt`, run `python spacex_dash_app.py`, and open `http://127.0.0.1:8050`.

## Visual analysis — answer the five lab questions

The cells below compute the results from the supplied CSV instead of assuming values from a screenshot.

In [ ]:
# 1–2. Successful count and success rate by launch site
site_summary = spacex_df.groupby('Launch Site').agg(
    launch_count=('class','size'), successful_count=('class','sum'), success_rate=('class','mean')
).sort_values('successful_count', ascending=False)
display(site_summary)
print('Largest successful count site:', site_summary['successful_count'].idxmax())
print('Highest success-rate site:', site_summary['success_rate'].idxmax())

In [ ]:
# 3–4. Launch success rate by 1,000 kg payload bands
payload_edges = list(range(0, max_payload + 1000, 1000))
payload_band = pd.cut(spacex_df['Payload Mass (kg)'], bins=payload_edges, right=False, include_lowest=True)
payload_summary = spacex_df.assign(Payload_Range=payload_band).groupby('Payload_Range', observed=False)['class'].agg(launches='size', success_rate='mean')
payload_summary = payload_summary[payload_summary['launches'] > 0]
display(payload_summary)
print('Highest success-rate payload band(s):')
display(payload_summary[payload_summary.success_rate == payload_summary.success_rate.max()])
print('Lowest success-rate payload band(s):')
display(payload_summary[payload_summary.success_rate == payload_summary.success_rate.min()])

In [ ]:
# 5. Success rate by exact booster version
booster_summary = spacex_df.groupby('Booster Version').agg(
    launches=('class','size'), successful_count=('class','sum'), success_rate=('class','mean')
).sort_values(['success_rate','launches'], ascending=[False,False])
display(booster_summary)
print('Highest observed booster-version success rate:', booster_summary['success_rate'].idxmax())

## Screenshots and GitHub submission

Run the Dash app in a browser, then capture and save these three live-dashboard screenshots in `screenshots/`:
- `dashboard_all_sites.png`
- `dashboard_selected_site.png`
- `dashboard_payload_filter.png`

Upload this notebook, the Python app, requirements file, README, dataset (if redistribution is permitted), and screenshots to your GitHub repository. Do not upload credentials or personal access tokens.